# Car Price Prediction - Model Training & Evaluation

**Project:** Used Car Price Prediction  
**Course:** Data Science Capstone - 3rd Year  
**Author:** [Your Name]  
**Date:** 2024

In [ ]:
# Setup
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.config import config
from src.data.preprocessing import run_preprocessing_pipeline
from src.features.build_features import build_features
from src.models.train import (
    train_all_models,
    select_best_model,
    save_model,
    train_with_optimization,
)
from src.models.predict import evaluate_predictions
from src.visualization.visualize import (
    generate_model_report,
    plot_model_comparison,
    plot_predicted_vs_actual,
    plot_residual_distribution,
    plot_feature_importance,
    plot_permutation_importance,
    plot_shap_summary,
)

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

print("Environment ready")

In [ ]:
# Load data and build features
train_df, test_df = run_preprocessing_pipeline()
X_train, X_test, y_train, y_test = build_features(train_df, test_df)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {list(X_train.columns)}")

In [ ]:
# Train all models with cross-validation
results = train_all_models(X_train, y_train, X_test, y_test)

# Select best model
best_name, best_model = select_best_model(results, metric='rmse_mean')

In [ ]:
# Model comparison visualization
fig = plot_model_comparison(results, 'rmse_mean', save=True)
plt.show()

fig = plot_model_comparison(results, 'r2_mean', save=True)
plt.show()

In [ ]:
# Hyperparameter optimization for best model
print(f"Optimizing {best_name}...")
optimized_model, opt_info = train_with_optimization(best_name, X_train, y_train, X_test, y_test)

# Compare optimized vs default
default_pred = results[best_name]['model'].predict(X_test)
opt_pred = optimized_model.predict(X_test)

print("\nDefault model:")
evaluate_predictions(y_test.values, default_pred, f"{best_name} (default)")

print("\nOptimized model:")
evaluate_predictions(y_test.values, opt_pred, f"{best_name} (optimized)")

# Use optimized if better
if opt_info.get('val_metrics', {}).get('rmse', float('inf')) < results[best_name]['val_metrics']['rmse']:
    best_model = optimized_model
    print("\nUsing optimized model!")
else:
    print("\nDefault model performs better, keeping it.")

In [ ]:
# Final evaluation on test set
y_pred = best_model.predict(X_test)
metrics = evaluate_predictions(y_test.values, y_pred, f"{best_name} (final)")

In [ ]:
# Save best model
save_model(best_model, 'best_model')

# Save all models for comparison
for name, result in results.items():
    save_model(result['model'], name)

In [ ]:
# Generate full model evaluation report
feature_names = list(X_train.columns)
report_figures = generate_model_report(
    results,
    best_name,
    best_model,
    X_test,
    y_test,
    feature_names
)

print("\nModel report generated. Check reports/figures/")

In [ ]:
# Error analysis
test_with_preds = test_df.copy()
test_with_preds['predicted_price'] = y_pred
test_with_preds['residual'] = test_with_preds[config.get('data.target_column')] - test_with_preds['predicted_price']
test_with_preds['abs_error'] = test_with_preds['residual'].abs()
test_with_preds['pct_error'] = (test_with_preds['abs_error'] / test_with_preds[config.get('data.target_column')]) * 100

# Worst predictions
worst = test_with_preds.nlargest(10, 'pct_error')
print("Top 10 worst predictions (by % error):")
display_cols = ['name', 'year', 'km_driven', 'fuel', 'transmission',
                config.get('data.target_column'), 'predicted_price', 'pct_error']
print(worst[display_cols].to_string(index=False))

In [ ]:
# Best predictions
best = test_with_preds.nsmallest(10, 'pct_error')
print("Top 10 best predictions (by % error):")
print(best[display_cols].to_string(index=False))

In [ ]:
# Residuals by car age
if 'car_age' in X_test.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    test_with_preds['car_age'] = X_test['car_age'].values
    sns.boxplot(data=test_with_preds, x='car_age', y='residual', ax=ax)
    ax.axhline(y=0, color='red', linestyle='--')
    ax.set_title('Residuals by Car Age')
    ax.set_xlabel('Car Age (years)')
    ax.set_ylabel('Residual (INR)')
    plt.tight_layout()
    from src.visualization.visualize import save_figure
    save_figure(fig, 'residuals_by_age')
    plt.show()

In [ ]:
# Residuals by brand (top 15 brands)
if 'brand' in test_with_preds.columns:
    top_brands = test_with_preds['brand'].value_counts().head(15).index
    brand_data = test_with_preds[test_with_preds['brand'].isin(top_brands)]

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=brand_data, x='brand', y='residual', ax=ax)
    ax.axhline(y=0, color='red', linestyle='--')
    ax.set_title('Residuals by Brand (Top 15)')
    ax.set_xlabel('Brand')
    ax.set_ylabel('Residual (INR)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    save_figure(fig, 'residuals_by_brand')
    plt.show()

## Model Training Summary

### Models Compared:
- Linear Regression (baseline)
- Ridge Regression (regularized linear)
- Random Forest (ensemble)
- XGBoost (gradient boosting)
- LightGBM (gradient boosting)
- CatBoost (gradient boosting, handles categoricals natively)

### Best Model: `{{ best_name }}`

### Test Set Performance:
- **RMSE**: {{ metrics['rmse']:.0f }} INR
- **MAE**: {{ metrics['mae']:.0f }} INR
- **R²**: {{ metrics['r2']:.4f }}
- **MAPE**: {{ metrics['mape']:.2f }}%

### Key Insights:
1. Tree-based models (XGBoost/LightGBM/CatBoost) outperform linear models
2. Feature engineering (car_age, km_per_year) significantly helps
3. Target encoding for brand prevents overfitting on high-cardinality categorical
4. Residuals show some heteroscedasticity (higher errors for expensive cars)

### Next Steps for Improvement:
- Try log-transform on target (price is right-skewed)
- Add more external data (brand reliability, fuel prices)
- Ensemble multiple models
- Collect more data for rare brands